# Dataset preprocessor

Deze notebook leest de ruwe catalogus in, voert de afgesproken opschoonstappen uit en schrijft een nieuwe CSV naar `datasets/processed/catalogus_opgeschoond.csv`. Het bestand in `datasets/raw` wordt niet aangepast.

Voer de codecellen van boven naar beneden uit. De invoer en uitvoer zijn relatief aan de projectmap.

## Keuzes en uitleg

- **Titels:** omgezet naar kleine letters, zoals afgesproken.
- **Type materiaal:** schrijfwijzen worden gestandaardiseerd. `Boek - thriller` en `Boek - roman` worden `Boek`; het subtype roman/thriller staat niet consequent als apart veld en wordt daarom niet als materiaaltype behouden.
- **Niveau:** `2F-3F` wordt `2F`, het minimale niveau. Andere niveaus blijven staan.
- **Tags:** `onderwerp`, `Themas_boek` en `trefwoorden` overlappen. Hun tags worden samengevoegd in `trefwoorden`, zonder duplicaten; de dubbele bronkolommen worden daarna verwijderd.
- **Genre:** Anne Frank wordt gecorrigeerd naar `informatief`. Andere lege genres blijven leeg, omdat ze niet betrouwbaar uit de titel of andere velden zijn af te leiden.
- **Doelgroep:** schrijfwijzen worden lowercase en spaties rond `/` verdwijnen. Lege doelgroepen, ook bij tijdschriften, blijven leeg: uit alleen een titel is MBO of NT2 niet betrouwbaar af te leiden.
- **Jaar en pagina's:** `onbekend`, 2099 en ontbrekende jaren blijven leeg. Negatieve paginatellingen worden leeg in plaats van 0; 0 zou een geldige maar onjuiste waarde suggereren.
- **Behouden filters:** `publicatiejaar` en `beschikbaar_formaat` blijven beschikbaar als mogelijke filters; formaatschrijfwijzen worden lowercase.
- **Beoordeling:** `Gemiddelde beoordeling` blijft staan als mogelijk aanbevelingssignaal. Lege waarden blijven leeg.
- **Verwijderde velden:** lege, dubbele, administratieve en niet-relevante tekstcomplexiteitsvelden worden verwijderd. `cataloguscode` blijft staan als unieke code.
- **Auteur:** niet automatisch aangepast of ingevuld; namen moeten daarvoor handmatig of met een betrouwbare bron worden gecontroleerd.

## 1. Ruwe dataset inlezen

De CSV is puntkomma-gescheiden en wordt expliciet als UTF-8 gelezen.

In [1]:
from pathlib import Path

import pandas as pd

source_path = Path("datasets") / "raw" / "catalogus.csv"
df = pd.read_csv(source_path, encoding="utf-8", sep=";")

print(f"Ingelezen: {source_path} ({df.shape[0]} rijen, {df.shape[1]} kolommen)")

Ingelezen: datasets\raw\catalogus.csv (283 rijen, 33 kolommen)


## 2. Titels en categorieën standaardiseren

Onbekende materiaaltypen worden niet stilzwijgend aangepast: de controle laat de cel stoppen zodat een nieuwe schrijfwijze bewust kan worden beoordeeld.

In [2]:
clean_df = df.copy()
clean_df["Titel"] = clean_df["Titel"].str.strip().str.lower()

material_map = {
    "boek": "Boek",
    "boek - thriller": "Boek",
    "boek - roman": "Boek",
    "online artikel": "Online artikel",
    "dichtbundel": "Dichtbundel",
    "blogpost": "Blogpost",
    "tijdschrift": "Tijdschrift",
    "krant (papier)": "Krant (papier)",
    "digitale krant": "Digitale krant",
}
material_type = clean_df["Type materiaal"].str.strip().str.casefold()
unknown_material_types = set(material_type.dropna().unique()) - set(material_map)
assert not unknown_material_types, f"Onbekende materiaaltypen: {unknown_material_types}"
clean_df["Type materiaal"] = material_type.map(material_map)

clean_df["Niveau"] = clean_df["Niveau"].str.strip().str.upper().replace({"2F-3F": "2F"})
clean_df["doelgroep"] = clean_df["doelgroep"].str.strip().str.lower().str.replace(
    r"\s*/\s*", "/", regex=True
)
clean_df["beschikbaar_formaat"] = clean_df["beschikbaar_formaat"].str.strip().str.lower()

## 3. Tags samenvoegen

Dezelfde tag kan met verschillende spaties of hoofdletters voorkomen. Vergelijking gebeurt daarom hoofdletterongevoelig; de eerste oorspronkelijke schrijfwijze blijft bewaard.

In [3]:
def merge_tags(row):
    tags = []
    seen = set()
    for value in row:
        if pd.isna(value):
            continue
        for tag in str(value).split(";"):
            tag = tag.strip()
            if tag.startswith(("http", "[")):
                continue            # URL of brontag zoals "[naturetoday.com]": komt uit Themas_boek, is geen trefwoord
            if tag and tag.casefold() not in seen:
                tags.append(tag)
                seen.add(tag.casefold())
    return "; ".join(tags) if tags else pd.NA

tag_columns = ["trefwoorden", "onderwerp", "Themas_boek"]
clean_df["trefwoorden"] = clean_df[tag_columns].apply(merge_tags, axis=1)

## 4. Bekende genrecorrectie

Anne Frank stond ten onrechte als fictie gemarkeerd. Alleen deze bekende fout wordt aangepast; andere ontbrekende genres blijven oningevuld om aannames te voorkomen.

In [4]:
anne_frank = clean_df["Titel"].eq("het dagboek van anne frank")
clean_df.loc[anne_frank, "genre"] = "informatief"

## 5. Jaren en paginatellingen controleren

`onbekend` en het onmogelijke jaar 2099 worden ontbrekende waarden. Een negatieve of ontbrekende paginatelling krijgt ook geen verzonnen vervangingswaarde. De controles signaleren onverwachte niet-numerieke waarden en fractionele aantallen pagina's. Beide kolommen worden als nullable `Int64` opgeslagen, zodat ontbrekende waarden behouden blijven zonder decimalen.

In [5]:
year_text = clean_df["publicatiejaar"].str.strip()
unknown_year = year_text.str.casefold().eq("onbekend")
years = pd.to_numeric(year_text.mask(unknown_year), errors="coerce")
assert years.notna().sum() == year_text.notna().sum() - unknown_year.sum(), (
    "Ongeldige publicatiejaarwaarde gevonden"
)
clean_df["publicatiejaar"] = years.mask(years.eq(2099)).astype("Int64")

pages = pd.to_numeric(clean_df["aantal_paginas"], errors="coerce")
assert pages.notna().sum() == clean_df["aantal_paginas"].notna().sum(), (
    "Ongeldige paginawaarde gevonden"
)
assert pages.dropna().mod(1).eq(0).all(), "Niet-gehele paginatelling gevonden"
clean_df["aantal_paginas"] = pages.mask(pages.lt(0)).astype("Int64")

## 6. Niet-benodigde kolommen verwijderen

De unieke `cataloguscode` en velden die bruikbaar kunnen zijn voor aanbevelingen of filters blijven behouden. De expliciete lijst maakt zichtbaar welke velden uit de uitvoer verdwijnen.

In [6]:
columns_to_drop = [
    "Unnamed: 3", "materiaal_id", "interne_batchcode", "administratieve_prioriteit",
    "genre_kopie", "interessecluster", "taal", "doelgroep_extra", "gegevensstatus",
    "uitgever", "bron_url", "onderwerp", "Themas_boek", "geschatte_leestijd_min",
    "woordenaantal_omschrijving", "gem_woordlengte", "gem_zinslengte_woorden",
    "aandeel_lange_woorden_pct", "aantal_zinnen", "leescomplexiteit_score",
]
clean_df = clean_df.drop(columns=columns_to_drop)

assert len(clean_df) == len(df), "Het aantal datasetrijen is onverwacht gewijzigd"
assert clean_df["cataloguscode"].is_unique, "cataloguscode is niet langer uniek"
print(f"Overgebleven kolommen: {clean_df.shape[1]}")

Overgebleven kolommen: 13


## 7. Opgeschoonde dataset opslaan

De map wordt aangemaakt als die nog niet bestaat. De bron blijft onaangeroerd; deze cel maakt of vervangt alleen de opgeschoonde uitvoer.

In [7]:
output_dir = Path("datasets") / "processed"
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "catalogus_opgeschoond.csv"
clean_df.to_csv(output_path, index=False, encoding="utf-8", sep=";")

print(f"Opgeslagen: {output_path} ({clean_df.shape[0]} rijen, {clean_df.shape[1]} kolommen)")

Opgeslagen: datasets\processed\catalogus_opgeschoond.csv (283 rijen, 13 kolommen)
